# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. The dataset documents protein abundance and modification analysis using mass spectrometry on extracellular vesicles from human mast cells.

### Dataset Source
The dataset is available as a Croissant schema via:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.


In [ ]:
# List all record sets, fields, and columns with their @id from the dataset
print("Record sets available in this dataset:")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}, number of fields: {len(rs.fields)}")

print("\nDetails for each record set:")
for rs in record_sets:
    print(f"\nRecord Set: {rs.name}\n  @id: {rs.id}")
    for f in rs.fields:
        # f is mlcroissant.metadata.Field
        col_str = f.column.id if f.column is not None else None
        print(f"    Field: {f.name} (@id: {f.id}, dataType: {f.data_type}, column: {col_str})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s were listed above.


In [ ]:
# For this dataset, we'll select the main protein record set (by its @id)
# Let's find the main record set id and load its records

# Step 1: Identify all record_set @id strings
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"List of record_set @id's: {record_set_ids}")

# Step 2: For this example, we pick the first record set as the main Record Set
main_record_set_id = record_set_ids[0]
print(f"\nUsing Record Set @id: {main_record_set_id}")

# Step 3: Load all records for this record set into a DataFrame
df = pd.DataFrame(dataset.records(record_set=main_record_set_id))
print(f"Columns available: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process the data using common EDA steps: filter records, normalize numeric fields, and group data. We'll choose sample numeric fields based on the listed columns.

In [ ]:
# Display numeric columns to select for analysis
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric columns in record set:", numeric_cols)

# Select a numeric field and group field for demonstration
if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field: {numeric_field}")
else:
    raise ValueError("No numeric field found in this record set for demonstration.")

# If there's a likely categorical/group field, pick it, otherwise skip grouping
possible_group_fields = [col for col in df.columns if any(s in col.lower() for s in ["type","group","sample","source","class","category"])]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    print(f"Using group field: {group_field}")

# EDA 1: Filter records where the numeric field > threshold (e.g., 10th percentile)
threshold = df[numeric_field].quantile(0.1) if df[numeric_field].dtype in ['float64','int64'] else 0
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (10th percentile): {filtered_df.shape[0]}")
display(filtered_df.head())

# EDA 2: Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First 5 normalized values for {numeric_field}:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# EDA 3: Group by categorical field if available, take the mean
if group_field is not None and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the main numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group if grouping available
if group_field is not None and group_field in filtered_df.columns and filtered_df[group_field].nunique() < 30:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we've used `mlcroissant` to load and explore the FAIR² mass spectrometry dataset, examining fields with their `@id`s, extracting the main record set, performing basic EDA, and visualizing key features. You can adjust this notebook to analyze more fields and record sets as needed for deeper investigation.

*For more on the Croissant metadata standard and advanced usage, see [the mlcroissant documentation](https://mlcommons.github.io/croissant/).*